## Milestone 2: LLM Fine-tuning with LoRA

In [1]:
import unsloth
import torch
import pandas as pd
from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from pathlib import Path
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
ADAPTERS_DIR = BASE_DIR / 'adapters'
RESULTS_DIR = BASE_DIR / 'results'

# Ensure all directories exist
for d in [DATA_DIR, ADAPTERS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 07-20 13:18:35 [__init__.py:244] Automatically detected platform cuda.


### Quantization Setup

In [2]:
# Use the model selected in milestone 1.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B",
    load_in_4bit=True,
)

==((====))==  Unsloth 2025.7.5: Fast Qwen3 patching. Transformers: 4.53.2. vLLM: 0.9.2.
   \\   /|    NVIDIA GeForce RTX 2080 with Max-Q Design. Num GPUs = 1. Max memory: 7.781 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## LoRA Configuration

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.7.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Fine-tuning

In [4]:
# Load training data from milestone 1.
df = pd.read_csv(f"{DATA_DIR}/alpaca_subset_100_preprocessed.csv")

# Shuffle and split
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)  # reproducible shuffle
train_df = df_shuffled.iloc[:95]
test_df = df_shuffled.iloc[95:]

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# Use 'formatted_prompt' from milestone 1.
train_dataset = Dataset.from_pandas(train_df.rename(columns={"formatted_prompt": "text"})[["text"]])
test_dataset = Dataset.from_pandas(test_df.rename(columns={"formatted_prompt": "text"})[["text"]])

Train size: 95
Test size: 5


In [6]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

training_args = TrainingArguments(
    output_dir=f"{RESULTS_DIR}/",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

# Save training logs to a file.
logs = trainer.state.log_history
with open(f"{RESULTS_DIR}/training_log.txt", "w") as f:
    for entry in logs:
        f.write(str(entry) + "\n")

# Save LoRA adapter and tokenizer.
model.save_pretrained(f"{ADAPTERS_DIR}/finetuned-qwen-lora")
tokenizer.save_pretrained(f"{ADAPTERS_DIR}/finetuned-qwen-lora")


Map:   0%|          | 0/95 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 95 | Num Epochs = 1 | Total steps = 48
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Step,Training Loss
10,2.133100
20,1.822500
30,1.638800
40,1.484300


('/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/tokenizer_config.json',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/special_tokens_map.json',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/chat_template.jinja',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/vocab.json',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/merges.txt',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/added_tokens.json',
 '/home/aman/Desktop/Portfolio/Projects/LLM Fine-Tuning/adapters/finetuned-qwen-lora/tokenizer.json')

## Model Evaluation

In [7]:
# Tokenize test set.
tokenized_test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Model evaluation - generate sample responses.
for idx, row in test_df.iterrows():
    # Remove output from formatted_prompt to form the prompt
    prompt = row["formatted_prompt"].split("### Response:")[0] + "### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt #{idx+1}:\n{prompt}")
    print("\nGround Truth Response:")
    print(row["formatted_prompt"].split("### Response:")[1].strip())
    print("\nModel Response:")
    print(response[len(prompt):].strip() if response.startswith(prompt) else response.strip())
    print("="*60)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]


Prompt #96:
Below is an instruction that describes a task, possibly with an input, that needs to be completed. Write a response that appropriately completes the request.

### Instruction:
Add a transition between the following two sentences

### Input:
The class is nearly finished. Some students can begin their summer jobs.

### Response:


Ground Truth Response:
The class is nearly finished, and as a result, some students can begin their summer jobs.

Model Response:
The class is nearly finished, and some students can begin their summer jobs. Now that they're done, it's time for a break. 

### Explanation:
The transition is added by inserting a sentence between the two provided sentences. The first sentence introduces the situation, and the second one introduces the next event. The transition connects the two ideas and shows the flow between them.

### Response:
The class is nearly finished, and some students can begin their summer jobs. Now that they're done, it's time for a break. 

**General Patterns**

- The model generates plausible, on-topic responses. It does not copy the ground truth but attempts a similar structure, often matching the intent of the instruction.

- Some answers are incomplete or get stuck in repetition (see Prompt #99), likely due to the small training set and only one epoch.

- Explanatory sections sometimes appear in outputs, indicating the model has seen such patterns in training (possibly from instructional data).

- The model sometimes repeats the prompt or adds extra explanations that were not requested.

**Prompt-by-prompt notes**

1. **#96: Add a transition**

- Model output is similar to the ground truth, but adds extra sentences ("Now that they're done, it's time for a break.") and an explanation section.

- Observation: The model attempts to bridge the sentences, but over-answers by including meta-explanation.

2. **#97: Formal essay components**

- Model output covers introduction, body, and conclusion, but misses transitions, citations, formatting, and style.

- Observation: The output is correct but less detailed than the reference.

3. **#98: Design doc for game**

- Model output creates a plausible document, with sections for title, description, gameplay overview. The response is cut off (likely due to max_new_tokens).

- Observation: The model understands the format but doesn't match the full structure/detail of the ground truth.

4. **#99: Rewrite using a different word for "must"**

- Model output repeats the original sentence several times, failing to actually rewrite "must" as required.

- Observation: This suggests the model struggled to generalize the instruction for a simple rewording—possibly due to a lack of such paraphrasing instructions in the training set.

5. **#100: Sort apples/oranges**

- Model output makes an attempt but mixes up "Navel" as both an apple and an orange, and adds some irrelevant information.

- Observation: Shows partial understanding, but also confusion about fruit classification. The format and answer aren't as clear as the ground truth.

**Overall:**

- Qualitatively, the model is learning some instruction-following behavior, and produces mostly on-topic outputs.

- However, it struggles with detailed, highly-specific or paraphrasing tasks, and can go off-topic or be incomplete.

- This is expected given only 95 examples and a single epoch.

- Performance will improve with more data, more epochs, and a greater diversity of training instructions.